# 01 Data Understanding: SAML-D Data Audit
CRISP-DM Phase 2: This notebook checks whether the raw data can be trusted before cleaning and analysis. Issues found here are fixed in 02_cleaning

In [ ]:
from google.colab import drive
from pathlib import Path
import os

if os.path.ismount("/content/drive"):
    print("Google Drive is already mounted.")
else:
    drive.mount("/content/drive")

Google Drive is already mounted.


## Project Setup and Data Access

This is to automatically create the required project folders to rerun all notebooks and download the SAML-D dataset from Google Drive if it is not already available.

## To reproduce the project:
Mount Google Drive with the cell above, run the cell below, then execute Notebooks 01–05 in order. Each notebook saves the files needed by the next.

The initial dataset is approximately 996 MB. It only needs to be downloaded once.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Defining the project folder
PROJECT = Path("/content/drive/MyDrive/INFO320_AML")

# Defining the required folders
RAW = PROJECT / "Data" / "Raw"
PROCESSED = PROJECT / "Data" / "Processed"
TABLEAU = PROJECT / "Tableau"

# Creating folders if they do not already exist
for folder in [RAW, PROCESSED, TABLEAU]:
    folder.mkdir(parents=True, exist_ok=True)

# Defining the original dataset location
RAW_FILE = RAW / "SAML-D.csv"

# Downloading the dataset if it is not already available
if not RAW_FILE.exists():
    print("Dataset not found. Downloading SAML-D from Google Drive...")

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "gdown"])

    import gdown

    gdown.download(
        id="19aEdn0iUsych_VUb6rHh941jKla_FQt7",
        output=str(RAW_FILE),
        quiet=False)

# Verifing the dataset was downloaded successfully
assert RAW_FILE.is_file(), "Dataset not found. Check Google Drive sharing permissions."

print("Project folders ready.")
print("Dataset ready.")
print(f"Dataset size: {RAW_FILE.stat().st_size / 1_000_000:,.1f} MB")

Project folders ready.
Dataset ready.
Dataset size: 996.2 MB


Inspecting the column names and the data structure

In [ ]:
import pandas as pd

preview = pd.read_csv(
    RAW_FILE,
    nrows=5,
    dtype="string",
    keep_default_na=False)

print("Column names:")
for column in preview.columns:
    print(f"- {column}")

print(f"\nNumber of columns: {len(preview.columns)}")

display(preview)

Column names:
- Time
- Date
- Sender_account
- Receiver_account
- Amount
- Payment_currency
- Received_currency
- Sender_bank_location
- Receiver_bank_location
- Payment_type
- Is_laundering
- Laundering_type

Number of columns: 12


,Time,Date,Sender_account,Receiver_account,Amount,Payment_currency,Received_currency,Sender_bank_location,Receiver_bank_location,Payment_type,Is_laundering,Laundering_type
0,10:35:19,2022-10-07,8724731955,2769355426,1459.15,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits
1,10:35:20,2022-10-07,1491989064,8401255335,6019.64,UK pounds,Dirham,UK,UAE,Cross-border,0,Normal_Fan_Out
2,10:35:20,2022-10-07,287305149,4404767002,14328.44,UK pounds,UK pounds,UK,UK,Cheque,0,Normal_Small_Fan_Out
3,10:35:21,2022-10-07,5376652437,9600420220,11895.0,UK pounds,UK pounds,UK,UK,ACH,0,Normal_Fan_In
4,10:35:21,2022-10-07,9614186178,3803336972,115.25,UK pounds,UK pounds,UK,UK,Cash Deposit,0,Normal_Cash_Deposits


Counting all transactions confirming the date range and counting the labels

In [ ]:
from collections import Counter

total_rows = 0
invalid_dates = 0
earliest_date = None
latest_date = None
label_counts = Counter()
typology_counts = Counter()

for chunk in pd.read_csv(
    RAW_FILE,
    usecols=["Date", "Is_laundering", "Laundering_type"],
    dtype="string",
    keep_default_na=False,
    chunksize=250_000):
    total_rows += len(chunk)

    dates = pd.to_datetime(
        chunk["Date"], format="%Y-%m-%d", errors="coerce")
    invalid_dates += int(dates.isna().sum())

    if dates.notna().any():
        chunk_start = dates.min()
        chunk_end = dates.max()
        earliest_date = (chunk_start if earliest_date is None
            else min(earliest_date, chunk_start))
        latest_date = (chunk_end if latest_date is None
            else max(latest_date, chunk_end))

    label_counts.update(chunk["Is_laundering"])
    typology_counts.update(chunk["Laundering_type"])

print(f"Total transactions: {total_rows:,}")
print("Earliest date:", earliest_date)
print("Latest date:", latest_date)
print(f"Missing or unparseable dates: {invalid_dates:,}")

print("\nOutcome-label counts:")
print(dict(label_counts))

print("\nTransaction counts by typology:")
display(pd.DataFrame(typology_counts.most_common(),
        columns=["Typology", "Transactions"]))

Total transactions: 9,504,852
Earliest date: 2022-10-07 00:00:00
Latest date: 2023-08-23 00:00:00
Missing or unparseable dates: 0

Outcome-label counts:
{'0': 9494979, '1': 9873}

Transaction counts by typology:


,Typology,Transactions
0,Normal_Small_Fan_Out,3477717
1,Normal_Fan_Out,2302220
2,Normal_Fan_In,2104285
3,Normal_Group,528351
4,Normal_Cash_Withdrawal,305031
5,Normal_Cash_Deposits,223801
6,Normal_Periodical,210526
7,Normal_Plus_Mutual,155041
8,Normal_Mutual,125335
9,Normal_Foward,42031


Some cleaning is needed here "Normal Foward" is misspelled, some seperators are inconsistent (Stacked Bipartite", "Deposit-Send", "Fan_In") and capitalization needs to be standardized.

Checking whether each laundering label agrees and is consistent with its typologies. Normal typologies should = 0 suspicious typologies should = 1

In [ ]:
label_parts = []

for chunk in pd.read_csv(
    RAW_FILE,
    usecols=["Laundering_type", "Is_laundering"],
    dtype="string",
    keep_default_na=False,
    chunksize=250_000):
    counts = pd.crosstab(chunk["Laundering_type"], chunk["Is_laundering"])
    label_parts.append(counts)

label_check = pd.concat(label_parts).groupby(level=0).sum()
label_check = label_check.fillna(0).astype(int)

display(label_check)

Is_laundering,0,1
Laundering_type,,
Behavioural_Change_1,0,394
Behavioural_Change_2,0,345
Bipartite,0,383
Cash_Withdrawal,0,1334
Cycle,0,382
Deposit-Send,0,945
Fan_In,0,364
Fan_Out,0,237
Gather-Scatter,0,354


All the labels in the dataset agree no label inconsistencies were found.

Counting transactions and laundering transactions for each month

to help decide which months to train the model on and which to test on

In [ ]:
# Collecting a monthly summary from each chunk.
monthly_parts = []

for chunk in pd.read_csv(
    RAW_FILE,
    usecols=["Date", "Is_laundering"],
    dtype={"Is_laundering": "int"},
    chunksize=250_000):
    # Take the year and month from dates such as "2022-10-07".
    chunk["Month"] = chunk["Date"].str[:7]

    # Count transactions and add up the 1s (laundering labels).
    summary = chunk.groupby("Month")["Is_laundering"].agg(
        Transactions="count",
        Laundering_transactions="sum")
    monthly_parts.append(summary)

# Combine the summaries and add together matching months.
monthly_audit = pd.concat(monthly_parts).groupby("Month").sum()

# Calculate the percentage labeled laundering.
monthly_audit["Laundering_pct"] = (
    monthly_audit["Laundering_transactions"]
    / monthly_audit["Transactions"]
    * 100)
display(monthly_audit.round(4))

print("Total transactions:", monthly_audit["Transactions"].sum())
print("Total laundering transactions:", monthly_audit["Laundering_transactions"].sum())

,Transactions,Laundering_transactions,Laundering_pct
Month,,,
2022-10,708654,694,0.0979
2022-11,897621,794,0.0885
2022-12,900341,907,0.1007
2023-01,908320,940,0.1035
2023-02,904806,946,0.1046
2023-03,908562,927,0.1020
2023-04,902883,962,0.1065
2023-05,917601,866,0.0944
2023-06,897243,1024,0.1141


Total transactions: 9504852
Total laundering transactions: 9873


Every month contains transactions labeled as laundering. I plan to use OCT 2022 - April 2023 as training data, May - June as Validation and July - AUG as final test

Checking Each column for any missing values

In [ ]:
missing_values = {"", "na", "n/a", "null", "nan", "none", "unknown"}
missing_counts = pd.Series(0, index=preview.columns)

for chunk in pd.read_csv(
    RAW_FILE, dtype="string", keep_default_na=False, chunksize=250_000
):
    for column in chunk.columns:
        values = chunk[column].str.strip().str.lower()
        missing_counts[column] += values.isin(missing_values).sum()

display(missing_counts.rename("Missing-value matches"))

,Missing-value matches
Time,0
Date,0
Sender_account,0
Receiver_account,0
Amount,0
Payment_currency,0
Received_currency,0
Sender_bank_location,0
Receiver_bank_location,0
Payment_type,0


All 12 columns across all transactions there were 0 blank data found and no unknown or N/A values, so no rows require removal or replacement

Checking the Amount column making sure entry is a number and counting how many rows are 0 or negative

In [ ]:
invalid_amounts = 0
zero_amounts = 0
negative_amounts = 0

for chunk in pd.read_csv(RAW_FILE, usecols=["Amount"], dtype="string", chunksize=250_000):
    amounts = pd.to_numeric(chunk["Amount"], errors="coerce")
    invalid_amounts += amounts.isna().sum()
    zero_amounts += amounts.eq(0).sum()
    negative_amounts += amounts.lt(0).sum()

print("Unreadable amounts:", invalid_amounts)
print("Zero amounts:", zero_amounts)
print("Negative amounts:", negative_amounts)

Unreadable amounts: 0
Zero amounts: 0
Negative amounts: 0


Every amount is a number and there are no 0 amounts and no negative amounts

Checking whether each date and time combine into a valid timestamp and whether all the transactions are in chronological order

In [ ]:
invalid_timestamps = 0
in_time_order = True
previous_end = None

for chunk in pd.read_csv(RAW_FILE, usecols=["Date", "Time"], dtype="string", chunksize=250_000):
    timestamps = pd.to_datetime(chunk["Date"] + " " + chunk["Time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    invalid_timestamps += timestamps.isna().sum()

    if not timestamps.is_monotonic_increasing:
        in_time_order = False
    if previous_end is not None and timestamps.iloc[0] < previous_end:
        in_time_order = False
    previous_end = timestamps.iloc[-1]

print("Invalid timestamps:", invalid_timestamps)
print("File is in chronological order:", in_time_order)

Invalid timestamps: 0
File is in chronological order: True


All transactions have valid timestamps and the file is in chronological order. During cleaning I will combine them into one timestamp column.

Checking for duplicate rows that have exact matches across all 12 columns because the dataset does not give each transaction a unique transaction ID

In [ ]:
import csv

duplicate_rows = 0
rows_checked = 0
previous_timestamp = None
seen_this_second = set()

with open(RAW_FILE, newline="") as file:
    reader = csv.reader(file)
    next(reader)

    for row in reader:
        rows_checked += 1
        timestamp = (row[1], row[0])

        if timestamp != previous_timestamp:
            seen_this_second.clear()
            previous_timestamp = timestamp

        record = tuple(row)
        if record in seen_this_second:
            duplicate_rows += 1
        seen_this_second.add(record)

print("Rows checked:", rows_checked)
print("Exact duplicate rows beyond the first occurrence:", duplicate_rows)

Rows checked: 9504852
Exact duplicate rows beyond the first occurrence: 0


There are 0 duplicate rows or exact matches throughout the dataset, so i can keep all transactions.

Checking how many unique accounts there are in the dataset and making sure Account ID's only contain digits

In [ ]:
account_ids = set()

for chunk in pd.read_csv(RAW_FILE, usecols=["Sender_account", "Receiver_account"],
                         dtype="string", keep_default_na=False, chunksize=250_000):
    account_ids.update(chunk["Sender_account"].unique())
    account_ids.update(chunk["Receiver_account"].unique())

ids = pd.Series(list(account_ids), dtype="string")
non_digit_ids = ids[~ids.str.fullmatch(r"[0-9]+")]

print("Distinct accounts:", len(ids))
print("Distinct IDs containing anything other than digits:", len(non_digit_ids))
print("Examples of those IDs:", non_digit_ids.head(5).tolist())

Distinct accounts: 855460
Distinct IDs containing anything other than digits: 0
Examples of those IDs: []


The dataset has 855,460 distinct account ID's across the 9,504,852 transactions. Every ID is only digits but im keeping the ID's as text to preserve the original format

Checking the currencies, bank locations, and payment types for inconsistent spelling, capitalization, and extra spaces or missing values

In [ ]:
category_columns = [
    "Payment_currency", "Received_currency",
    "Sender_bank_location", "Receiver_bank_location",
    "Payment_type"]

category_values = {column: set() for column in category_columns}

for chunk in pd.read_csv(RAW_FILE, usecols=category_columns, dtype="string",
                         keep_default_na=False, chunksize=250_000):
    for column in category_columns:
        category_values[column].update(chunk[column].unique())

for column in category_columns:
    print(f"\n{column} ({len(category_values[column])} distinct values)")
    print(sorted(category_values[column]))


Payment_currency (13 distinct values)
['Albanian lek', 'Dirham', 'Euro', 'Indian rupee', 'Mexican Peso', 'Moroccan dirham', 'Naira', 'Pakistani rupee', 'Swiss franc', 'Turkish lira', 'UK pounds', 'US dollar', 'Yen']

Received_currency (13 distinct values)
['Albanian lek', 'Dirham', 'Euro', 'Indian rupee', 'Mexican Peso', 'Moroccan dirham', 'Naira', 'Pakistani rupee', 'Swiss franc', 'Turkish lira', 'UK pounds', 'US dollar', 'Yen']

Sender_bank_location (18 distinct values)
['Albania', 'Austria', 'France', 'Germany', 'India', 'Italy', 'Japan', 'Mexico', 'Morocco', 'Netherlands', 'Nigeria', 'Pakistan', 'Spain', 'Switzerland', 'Turkey', 'UAE', 'UK', 'USA']

Receiver_bank_location (18 distinct values)
['Albania', 'Austria', 'France', 'Germany', 'India', 'Italy', 'Japan', 'Mexico', 'Morocco', 'Netherlands', 'Nigeria', 'Pakistan', 'Spain', 'Switzerland', 'Turkey', 'UAE', 'UK', 'USA']

Payment_type (7 distinct values)
['ACH', 'Cash Deposit', 'Cash Withdrawal', 'Cheque', 'Credit card', 'Cross-

There are 13 currency labels, 18 bank locations, and 7 payment types. The currency labels match across the payment and received columns and the location labels also match across the sender and receiver columns. There appears to be no obvious spelling variations, extra spaces or missing values.

The dataset lists both "Dirham" and "Moroccan dirham" separately. I'm comparing currencies with the receiver bank location to check whether they represent distinct currencies or are inconsistent names for the same currency

In [ ]:
dirham_parts = []

for chunk in pd.read_csv(RAW_FILE,
    usecols=["Received_currency", "Receiver_bank_location"],
    dtype="string",
    keep_default_na=False,
    chunksize=250_000):
    selected = chunk[chunk["Received_currency"].isin(["Dirham", "Moroccan dirham"])]
    counts = pd.crosstab(selected["Receiver_bank_location"], selected["Received_currency"])
    dirham_parts.append(counts)

dirham_check = pd.concat(dirham_parts).groupby(level=0).sum()
dirham_check = dirham_check.fillna(0).astype(int)

display(dirham_check)

Received_currency,Dirham,Moroccan dirham
Receiver_bank_location,,
Albania,32,35
Austria,27,28
France,32,30
Germany,31,34
India,39,28
Italy,21,35
Japan,35,28
Mexico,32,25
Morocco,30,38307


"Dirham" transactions were concentrated at UAE banks while "Moroccan dirham" transactions were concentrated at Moroccan banks supporting that they represent two distinct currencies.